In [1]:
import pandas as pd
import seaborn as sns
import glob
import statsmodels.api as sm

import altair as alt
#from vega_datasets import data

#source = data.seattle_weather()

In [2]:
# compare the row group size
sizes = ['1k','4k','32k','128k','1024k']

In [3]:
def sum_df(raw_df):
    return raw_df.query('id==438601312892825').groupby(["radius_meter","prec"])\
    .agg({'query_time':'min','join_time':'mean','valid':'first'}).reset_index()\
    .rename(columns={'prec':'depth','valid':'features'})

In [4]:
source_dict = {}
for sz in sizes:
    source_dict[sz] = pd.concat([pd.read_csv(f"{fn}") for fn in glob.glob(f"./output/{sz}/perf_parquet*.csv")])
    source_dict[sz] = sum_df(source_dict[sz])
    source_dict[sz]['row_group'] = sz

In [5]:
sources_stack = pd.concat([source_dict[sz] for sz in sizes])

In [6]:
sources_stack

,radius_meter,depth,query_time,join_time,features,row_group
0,100,4,8.687085,0.048919,15,1k
1,200,4,8.526292,0.009094,31,1k
2,300,4,8.532993,0.010077,37,1k
3,400,4,8.607641,0.009018,51,1k
4,500,4,8.390351,0.008902,56,1k
...,...,...,...,...,...,...
30,250000,4,42.062437,1.473324,535865,1024k
31,300000,4,53.519207,1.976909,664422,1024k
32,350000,4,68.775071,2.642833,872788,1024k
33,400000,4,111.718220,3.011082,1257155,1024k


In [10]:
alt.Chart(sources_stack.query("depth==4 ")).mark_line(interpolate="monotone").encode(
    x=alt.X("radius_meter:Q").title('query range (m)'), #.scale(type="log"),
    y=alt.Y("query_time:Q").title('query time (secs)'),
    color=alt.Color("row_group:N", legend=alt.Legend(title="row group size"))
).properties(
    width=600,
    height=200
)

alt.Chart(...)

In [8]:
alt.Chart(sources_stack.query("depth==4 and radius_meter < 100000")).mark_line(interpolate="monotone").encode(
    x=alt.X("radius_meter:Q").title('query range (m)'), #.scale(type="log"),
    y=alt.Y("query_time:Q").title('query time (secs)'),
    color=alt.Color("row_group:N", legend=alt.Legend(title="row group size"))
).properties(
    width=600,
    height=200
)

alt.Chart(...)

In [9]:
alt.Chart(sources_stack.query("depth==4 and radius_meter >= 100000")).mark_line(interpolate="monotone").encode(
    x=alt.X("radius_meter:Q").title('query range (m)'), #.scale(type="log"),
    y=alt.Y("query_time:Q").title('query time (secs)'),
    color=alt.Color("row_group:N", legend=alt.Legend(title="row group size"))
).properties(
    width=600,
    height=200
)

alt.Chart(...)

In [3]:
#load in results by different test profile

In [35]:
profiles = ['geojson_prepartition','geojson_offset','parquet_prepartition','parquet_offset']

In [36]:
folders = ['gj_prepar','Jun3','Jun4','Jul12']
ftypes = ['geojson','geojson','parquet','parquet']

In [38]:
source_dict = {}
for i in range(4):
    p,d,t = profiles[i],folders[i],ftypes[i]
    source_dict[p] = pd.concat([pd.read_csv(f"{fn}") for fn in glob.glob(f"./output/{d}/perf_{t}*.csv")])
    source_dict[p] = sum_df(source_dict[p])
    source_dict[p]['indexing'] = p

In [39]:
sources_stack = pd.concat([source_dict[p] for p in profiles])

In [40]:
alt.Chart(sources_stack.query("depth==4")).mark_line(interpolate="monotone").encode(
    x=alt.X("radius_meter:Q").title('query range (m)'), #.scale(type="log"),
    y=alt.Y("query_time:Q").title('query time (secs)'),
    color=alt.Color("indexing:N", legend=alt.Legend(title="type of index"))
).properties(
    width=600,
    height=200
)

alt.Chart(...)

In [29]:
alt.Chart(sources_stack.query("depth==4 and radius_meter < 100000")).mark_line(interpolate="monotone").encode(
    x=alt.X("radius_meter:Q").title('query range (m)'), #.scale(type="log"),
    y=alt.Y("query_time:Q").title('query time (secs)'),
    color=alt.Color("indexing:N", legend=alt.Legend(title="type of index"))
).properties(
    width=600,
    height=200
)

alt.Chart(...)

In [2]:
# extended range query

In [3]:
profiles = ['geojson_prepartition','geojson_offset','parquet_prepartition','parquet_offset']

In [4]:
folders = ['ext_js_prep','ext_js_offset','ext_pq_prep','ext_pq_offset']
ftypes = ['geojson','geojson','parquet','parquet']

In [7]:
source_dict = {}
for i in range(4):
    p,d,t = profiles[i],folders[i],ftypes[i]
    source_dict[p] = pd.concat([pd.read_csv(f"{fn}") for fn in glob.glob(f"./output/{d}/perf_{t}*.csv")])
    source_dict[p] = sum_df(source_dict[p])
    source_dict[p]['indexing'] = p

In [8]:
sources_stack = pd.concat([source_dict[p] for p in profiles])

In [10]:
alt.Chart(sources_stack.query("depth==4 and radius_meter < 3000000")).mark_line(interpolate="monotone").encode(
    x=alt.X("radius_meter:Q").title('query range (m)'), #.scale(type="log"),
    y=alt.Y("query_time:Q").title('query time (secs)'),
    color=alt.Color("indexing:N", legend=alt.Legend(title="type of index"))
).properties(
    width=600,
    height=200
)

alt.Chart(...)

In [13]:
src = sources_stack.query("depth==4 and radius_meter < 3000000")

In [15]:
src['percentage'] = 100* src['features'] / 13800000

/tmp/ipykernel_236608/2082107334.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  src['percentage'] = 100* src['features'] / 13800000


In [ ]:
src

In [19]:
lines = alt.Chart(src).mark_line(interpolate="monotone",point=True).encode(
    x=alt.X("percentage:Q").title('percentage of total features'), #.scale(type="log"),
    y=alt.Y("query_time:Q").title('query time (secs)'),
    color=alt.Color("indexing:N", legend=alt.Legend(title="type of index"))
).properties(
    width=600,
    height=200
)

In [26]:
rule_gj =  (
    alt.Chart().mark_rule(strokeDash=[12, 6], size=2, color='black').encode(y=alt.datum(1768))
)
rule_pq =  (
    alt.Chart().mark_rule(strokeDash=[6, 12], size=2, color='black').encode(y=alt.datum(144))
)

In [27]:
lines+rule_gj+rule_pq

alt.LayerChart(...)

In [4]:
source_gj_cid = pd.concat([pd.read_csv(f"{fn}") for fn in glob.glob("./output/gj_prepar/perf_geojson*.csv")])
source_gj_off = pd.concat([pd.read_csv(f"{fn}") for fn in glob.glob("./output/Jun3/perf_geojson*.csv")])
source_pq_off = pd.concat([pd.read_csv(f"{fn}") for fn in glob.glob("./output/Jul12/perf_parquet*.csv")])
source_pq_off_4k = pd.concat([pd.read_csv(f"{fn}") for fn in glob.glob("./output/Jul14_4k/perf_parquet*.csv")])

source_pq_cid = pd.concat([pd.read_csv(f"{fn}") for fn in glob.glob("./output/Jun4/perf_parquet*.csv")])

In [ ]:
## plot 1 
source = pd.read_csv("../assets/clean_size.csv")
source.dtypes

base = alt.Chart(source).encode(
    alt.X('dataset_size:N').title('dataset size')
).properties(
    width=300,
    height=200
)


line1 = base.mark_line(stroke='#5276A7', interpolate='monotone').encode(
    alt.Y('query_time', axis=alt.Axis(title='query time (secs)',titleColor='#5276A7'))
)

line2 = base.mark_line(stroke='orange', interpolate='monotone').encode(
    alt.Y('time_per_feature', axis=alt.Axis(title='query time per feature (secs)',titleColor='orange'))
)

alt.layer(line2, line1).resolve_scale(
    y='independent'
)

In [37]:
source = pd.read_csv("../assets/query_size.csv")
source.dtypes

radius      float64
features      int64
time        float64
depth         int64
dtype: object

In [40]:
source

,radius,features,time,depth
0,0.005,9,0.42,5
1,0.008,29,0.43,5
2,0.010,119,0.45,5
3,0.030,1799,2.25,5
4,0.080,15699,3.41,5
...,...,...,...,...
58,8.000,4711558,1317.58,4
59,9.000,5204836,1590.89,4
60,10.000,5728332,1962.84,4
61,11.000,6146501,2212.57,4


In [ ]:
## plot fitted line

In [7]:
def sum_df(raw_df):
    return raw_df.query('id==438601312892825').groupby(["radius_meter","prec"])\
    .agg({'query_time':'min','join_time':'mean','valid':'first'}).reset_index()\
    .rename(columns={'prec':'depth','valid':'features'})

In [ ]:
for sz in si

In [7]:
src1 = sum_df(source_gj_off)
src1['indexing'] = 'geojson_offset'
src2 = sum_df(source_gj_cid)
src2['indexing'] = 'geojson_prepartition'
src3 = sum_df(source_pq_off)
src3['indexing'] = 'parquet_offset'
src4 = sum_df(source_pq_cid)
src4['indexing'] = 'parquet_prepartition'

In [8]:
src2

,radius_meter,depth,query_time,join_time,features,indexing
0,100,4,1.728926,0.021799,15,geojson_prepartition
1,200,4,1.799593,0.012446,31,geojson_prepartition
2,300,4,1.782846,0.010670,37,geojson_prepartition
3,400,4,1.773583,0.010238,51,geojson_prepartition
4,500,4,1.771888,0.012135,56,geojson_prepartition
5,600,4,1.779838,0.010535,83,geojson_prepartition
6,700,4,1.749604,0.012534,94,geojson_prepartition
7,800,4,1.719849,0.014588,105,geojson_prepartition
8,900,4,1.732850,0.011953,118,geojson_prepartition
9,1000,4,1.728863,0.015750,137,geojson_prepartition


In [ ]:
src5 = sum_df(source_pq_off_4k)
src5['indexing'] = 'parquet_offset_4k'

In [74]:
source = src1

In [11]:
source = pd.concat([src1,src2,src3])

In [176]:
source = pd.concat([src2,src4])

In [12]:
source

,radius_meter,depth,query_time,join_time,features,indexing
0,100,3,26.814481,0.105675,15,geojson
1,100,4,1.538913,0.009459,15,geojson
2,100,5,0.303126,0.006509,15,geojson
3,200,3,25.605615,0.101733,31,geojson
4,200,4,1.506519,0.008881,31,geojson
...,...,...,...,...,...,...
100,400000,4,19.299514,2.883812,1257155,parquet_cid
101,400000,5,896.611101,3.053157,1257155,parquet_cid
102,450000,3,5.658935,3.615004,1507937,parquet_cid
103,450000,4,23.859041,3.339626,1507937,parquet_cid


In [27]:
ols_df = source.query('depth == 4 and indexing == "parquet_cid" and radius_meter > 99999')

ols_df['total'] = ols_df.eval('query_time+join_time')
Yc = ols_df['total']
X = ols_df['features']
X = sm.add_constant(X)

/tmp/ipykernel_236931/253477479.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ols_df['total'] = ols_df.eval('query_time+join_time')


In [28]:
model = sm.OLS(Yc,X)
results = model.fit()
results.params

const      -0.165790
features    0.000019
dtype: float64

In [46]:
ols_df = source.query('depth == 4 and indexing == "parquet_offset" and radius_meter > 99999')

ols_df['total'] = ols_df.eval('query_time+join_time')
Yo = ols_df['total']
X = ols_df['features']
X = sm.add_constant(X)

/tmp/ipykernel_236931/1402502285.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ols_df['total'] = ols_df.eval('query_time+join_time')


In [30]:
(100*X.features / 13800000).values

array([ 2.0048913 ,  2.55492029,  3.26849275,  3.88307971,  4.81465217,
        6.32455072,  9.10981884, 10.92707971])

In [31]:
model = sm.OLS(Yo,X)
results2 = model.fit()
results2.params

const      -12.183145
features     0.000089
dtype: float64

In [41]:
pX = range(1,101,1)
ypred = results.predict([[1,13800000*x/100] for x in pX])
ypred2 = results2.predict([[1,13800000*x/100] for x in pX])

In [36]:
px = (100*X.features / 13800000).values
px

array([ 2.0048913 ,  2.55492029,  3.26849275,  3.88307971,  4.81465217,
        6.32455072,  9.10981884, 10.92707971])

In [37]:
Yc

82      2.973308
85      5.010954
88      6.949915
91      9.372218
94     19.514965
97     17.985166
100    22.183326
103    27.198667
Name: total, dtype: float64

In [38]:
Yo

82      10.834653
85      14.264721
88      36.492022
91      31.241679
94      50.382723
97      66.770804
100     94.377332
103    124.100717
Name: total, dtype: float64

In [42]:
lines = (
    alt.Chart(pd.DataFrame({'pct':pX,'time_estimate':ypred}))
    .mark_line()
    .encode(x="pct", y="time_estimate" )
)



yrule = (
    alt.Chart().mark_rule(strokeDash=[12, 6], size=2).encode(y=alt.datum(1768))
)


lines + yrule

alt.LayerChart(...)

In [43]:
pd.DataFrame({'pct':pX,'offset':ypred,'cid':ypred2})\
              .melt(id_vars='pct',value_vars=['offset','cid'],var_name="indexing",value_name="time_estimate")

,pct,indexing,time_estimate
0,1,offset,2.457699
1,2,offset,5.081188
2,3,offset,7.704676
3,4,offset,10.328165
4,5,offset,12.951654
...,...,...,...
195,96,cid,1165.066147
196,97,cid,1177.329161
197,98,cid,1189.592174
198,99,cid,1201.855187


In [54]:
lines = (
    alt.Chart(pd.DataFrame({'pct':pX,'pre-partition':ypred,'offset-length':ypred2})\
              .melt(id_vars='pct',value_vars=['offset-length','pre-partition'],var_name="indexing",value_name="time_estimate"))
    .mark_line()
    .encode( x = "pct",
            #x=alt.X("pct",scale=alt.Scale(type='log')),
            y="time_estimate",
            #y= alt.Y("time_estimate",scale=alt.Scale(type='log')),
            color="indexing" )
)

dots = alt.Chart(pd.DataFrame({'pct':px,'offset-length':Yo,'pre-partition':Yc})\
                 .melt(id_vars='pct',value_vars=['offset-length','pre-partition'],var_name="indexing",value_name="time_used")
                ).mark_point().encode(
    x="pct:Q",
    y="time_used:Q",color="indexing"
)

yrule = (
    alt.Chart().mark_rule(strokeDash=[12, 6], size=2).encode(y=alt.datum(144))
)


lines + yrule + dots

alt.LayerChart(...)

In [101]:
ypred

array([ 24.54823477,  49.26225936,  73.97628395,  98.69030853,
       123.40433312, 148.11835771, 172.8323823 , 197.54640688,
       222.26043147])

In [ ]:
const        10.603498
education     0.594859
dtype: float64

In [179]:
source

,radius_meter,depth,query_time,join_time,features,idx
0,100,3,2.336353,0.053962,15,parquet_offset
1,100,4,0.576339,0.020220,15,parquet_offset
2,100,5,0.438677,0.017931,15,parquet_offset
3,200,3,2.336965,0.055180,31,parquet_offset
4,200,4,0.530244,0.015209,31,parquet_offset
...,...,...,...,...,...,...
100,400000,4,93.390674,2.599588,1257155,parquet_offset_4k
101,400000,5,385.326331,2.481288,1257155,parquet_offset_4k
102,450000,3,36.363995,3.348233,1507937,parquet_offset_4k
103,450000,4,102.800014,3.027527,1507937,parquet_offset_4k


In [181]:
source.query('depth==3')

,radius_meter,depth,query_time,join_time,features,idx
0,100,3,2.336353,0.053962,15,parquet_offset
3,200,3,2.336965,0.055180,31,parquet_offset
6,300,3,2.427019,0.065697,37,parquet_offset
9,400,3,2.349468,0.066620,51,parquet_offset
12,500,3,2.392827,0.059173,56,parquet_offset
...,...,...,...,...,...,...
90,250000,3,14.037634,1.168982,535865,parquet_offset_4k
93,300000,3,29.492378,1.690126,664422,parquet_offset_4k
96,350000,3,24.391626,2.461847,872788,parquet_offset_4k
99,400000,3,28.013046,2.777023,1257155,parquet_offset_4k


In [198]:
alt.Chart(source.query("depth==4")).mark_line(interpolate="monotone").encode(
    x=alt.X("features:Q").title('query feature size'), #.scale(type="log"),
    y=alt.Y("query_time:Q").title('query time (secs)'),
    color=alt.Color("indexing:N", legend=alt.Legend(title="type of index"))
).properties(
    width=600,
    height=200
)

alt.Chart(...)

In [ ]:
alt.Chart(source.query('radius < 4')).mark_line(interpolate="monotone").encode(
    x=alt.X("features:Q").title('query feature size'), #.scale(type="log"),
    y=alt.Y("time:Q").title('query time (secs)'),
    color="depth:N"
).properties(
    width=600,
    height=200
)

In [41]:
alt.Chart(source.query('radius < 4')).mark_line(interpolate="monotone").encode(
    x=alt.X("features:Q").title('query feature size'), #.scale(type="log"),
    y=alt.Y("time:Q").title('query time (secs)'),
    color="depth:N"
).properties(
    width=600,
    height=200
)

alt.Chart(...)

In [44]:
chart = alt.Chart(source.query('depth == 4')).mark_line(interpolate="monotone").encode(
    x=alt.X("features:Q").title('query feature size').scale(type="log"),
    y=alt.Y("time:Q").title('query time (secs)')
).properties(
    width=600,
    height=200
)

line = alt.Chart(pd.DataFrame({'y': [1800]})).mark_rule().encode(y='y')
chart+line

alt.LayerChart(...)

In [45]:
source.query('depth == 4')

,radius,features,time,depth
27,0.005,9,12.46,4
28,0.008,29,12.10,4
29,0.010,119,11.91,4
30,0.030,1799,14.91,4
31,0.080,15699,17.00,4
32,0.100,31459,23.68,4
33,0.300,126783,32.17,4
34,0.400,158169,41.37,4
35,0.800,278941,59.02,4
36,1.000,314746,61.72,4
